In [12]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import RobustScaler, OneHotEncoder, LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

In [13]:
## Loading the dataset
df = pd.read_csv('./Churn_Modelling.csv')
df.head(10)

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
5,6,15574012,Chu,645,Spain,Male,44,8,113755.78,2,1,0,149756.71,1
6,7,15592531,Bartlett,822,France,Male,50,7,0.00,2,1,1,10062.80,0
7,8,15656148,Obinna,376,Germany,Female,29,4,115046.74,4,1,0,119346.88,1
8,9,15792365,He,501,France,Male,44,4,142051.07,2,0,1,74940.50,0
9,10,15592389,H?,684,France,Male,27,2,134603.88,1,1,1,71725.73,0


In [14]:
## Dropping unnecessary columns
df = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CreditScore      10000 non-null  int64  
 1   Geography        10000 non-null  object 
 2   Gender           10000 non-null  object 
 3   Age              10000 non-null  int64  
 4   Tenure           10000 non-null  int64  
 5   Balance          10000 non-null  float64
 6   NumOfProducts    10000 non-null  int64  
 7   HasCrCard        10000 non-null  int64  
 8   IsActiveMember   10000 non-null  int64  
 9   EstimatedSalary  10000 non-null  float64
 10  Exited           10000 non-null  int64  
dtypes: float64(2), int64(7), object(2)
memory usage: 859.5+ KB


In [16]:
## Creating Label Encoders for Categorical Columns: "Gender"
le = LabelEncoder()
df['Gender'] = le.fit_transform(df['Gender'])

## Creating One Hot Encoders for Categorical Columns: "Geography"
ohe = OneHotEncoder(sparse_output=False)
encoded_geography = ohe.fit_transform(df[['Geography']])
geography_df = pd.DataFrame(encoded_geography, columns=ohe.get_feature_names_out(['Geography']))

## Concatinating the OneHotEncoded DataFrame with the original DataFrame
df = pd.concat([df.drop('Geography', axis=1),geography_df], axis=1)
print("Current shape of the dataframe:", df.shape) 
display(df.head(10))

Current shape of the dataframe: (10000, 13)


,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
5,645,1,44,8,113755.78,2,1,0,149756.71,1,0.0,0.0,1.0
6,822,1,50,7,0.00,2,1,1,10062.80,0,1.0,0.0,0.0
7,376,0,29,4,115046.74,4,1,0,119346.88,1,0.0,1.0,0.0
8,501,1,44,4,142051.07,2,0,1,74940.50,0,1.0,0.0,0.0
9,684,1,27,2,134603.88,1,1,1,71725.73,0,1.0,0.0,0.0


In [17]:
## Splitting the dataset into features and target variable
X = df.drop('Exited', axis=1)
y = df['Exited']

## Splitting the dataset into Training and Testing sets
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2,random_state=42)
print("Shape of splitted Data:", [X_train.shape, X_test.shape, y_train.shape, y_test.shape])

## Scalling the Train & Test set
scaler = StandardScaler()
# scaler = RobustScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
print("Scaling completed.")

Shape of splitted Data: [(8000, 12), (2000, 12), (8000,), (2000,)]
Scaling completed.


## Using Scikeras.wrappers

In [18]:
def create_model(neurons=16, layers=1):
    model = Sequential()
    model.add(Dense(neurons, activation='relu', input_shape=(X_train.shape[1],)))

    for _ in range(layers - 1):
        model.add(Dense(neurons,activation='relu'))

    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    return model

In [19]:
## Creating the Keras Classifier object
model = KerasClassifier(build_fn=create_model, epochs=20, batch_size=10, verbose =0,layers=1,neurons=16)


## Defining the Hyperparameter grid
param_grid = {
    'neurons': [32, 64, 128],
    'layers': [1, 2],
    # 'batch_size': [10, 20],
    'epochs': [20, 30, 40, 50]
}

In [20]:
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

gpus = tf.config.list_physical_devices('GPU')
display(tf.config.experimental.get_memory_info('GPU:0'))
tf.keras.backend.clear_session()

{'current': 0, 'peak': 0}

In [21]:
## Perform GridSerchCV
grid = GridSearchCV(estimator=model, param_grid=param_grid, cv=3, n_jobs=-1)
grid_result = grid.fit(X_train, y_train)

print("Best Accuracy: {:.4f} using {}".format(grid_result.best_score_, grid_result.best_params_))

2026-01-12 20:21:17.035613: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-12 20:21:17.072205: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-12 20:21:17.074833: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-12 20:21:17.077887: I tensorflow/core/platform/cpu_featu

Best Accuracy: 0.8586 using {'epochs': 20, 'layers': 1, 'neurons': 32}


## Using Keras-tuner

* Hyperparameters
    1. How many number of hidden layers we should have?
    2. How many number of neurons we should have in hidden layers?
    3. Learning Rate

In [40]:
import keras_tuner as kt

def build_model(hp):
    model = Sequential()
    for i in range(hp.Int('num_layers', 2, 20)):
        model.add(Dense(units=hp.Int('units_' + str(i),
                                            min_value=32,
                                            max_value=512,
                                            step=32),
                               activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            hp.Choice('learning_rate', [1e-2, 1e-3, 1e-4])),
        loss='binary_crossentropy',
        metrics=['accuracy'])
    return model

In [41]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=5,
    executions_per_trial=3,
    directory='project',
    project_name='Customer_Churn_ANN_Tuning')

In [42]:
tuner.search(X_train, y_train,
             epochs=50,
             validation_data=(X_test, y_test),
             callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10)])

Trial 5 Complete [00h 00m 40s]
val_accuracy: 0.8645000060399374

Best val_accuracy So Far: 0.8664999802907308
Total elapsed time: 00h 03m 06s


In [43]:
tuner.results_summary()

Results summary
Results in project/Customer_Churn_ANN_Tuning
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 3 summary
Hyperparameters:
num_layers: 4
units_0: 480
units_1: 192
learning_rate: 0.01
units_2: 352
units_3: 352
units_4: 224
units_5: 160
units_6: 128
units_7: 416
units_8: 224
units_9: 288
units_10: 192
units_11: 32
units_12: 384
units_13: 32
units_14: 416
units_15: 352
units_16: 320
units_17: 128
Score: 0.8664999802907308

Trial 4 summary
Hyperparameters:
num_layers: 6
units_0: 512
units_1: 352
learning_rate: 0.01
units_2: 192
units_3: 192
units_4: 448
units_5: 512
units_6: 320
units_7: 256
units_8: 416
units_9: 128
units_10: 160
units_11: 96
units_12: 32
units_13: 64
units_14: 448
units_15: 32
units_16: 416
units_17: 192
Score: 0.8645000060399374

Trial 0 summary
Hyperparameters:
num_layers: 7
units_0: 160
units_1: 192
learning_rate: 0.01
units_2: 32
units_3: 32
units_4: 32
units_5: 32
units_6: 32
Score: 0.8636666735013326

Trial 1 summary
Hyper

In [44]:
top_params = tuner.get_best_hyperparameters(num_trials=1)
top_params[0].values

{'num_layers': 4,
 'units_0': 480,
 'units_1': 192,
 'learning_rate': 0.01,
 'units_2': 352,
 'units_3': 352,
 'units_4': 224,
 'units_5': 160,
 'units_6': 128,
 'units_7': 416,
 'units_8': 224,
 'units_9': 288,
 'units_10': 192,
 'units_11': 32,
 'units_12': 384,
 'units_13': 32,
 'units_14': 416,
 'units_15': 352,
 'units_16': 320,
 'units_17': 128}

In [45]:
best_model = tuner.get_best_models()[0]

/home/prashant/.conda/envs/deeplearning/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [46]:
import pickle

## Loading scaler model
with open('scaler.pkl','rb') as f:
    scaler = pickle.load(f)

## Loading gender encoder model
with open('encoder_gender.pkl','rb') as f:
    gender_encoder = pickle.load(f)

## Loading Geography encoder model
with open('encoder_geography.pkl','rb') as f:
    geography_encoder = pickle.load(f)

In [47]:
## Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}


## Create a function to preprocess the input data
def preprocess_input(data, gender_encoder=gender_encoder, geography_encoder=geography_encoder, scaler=scaler):

    '''Preprocess the input data for prediction.
    Args:
        data (dict): A dictionary containing the input features.
        gender_encoder (LabelEncoder): Fitted LabelEncoder for Gender.
        geography_encoder (OneHotEncoder): Fitted OneHotEncoder for Geography.
        scaler (StandardScaler): Fitted StandardScaler for scaling the data.
    Returns:
        np.ndarray: Preprocessed and scaled input data ready for prediction.
    '''

    ## Covert the data into a DataFrame
    input_df = pd.DataFrame([data])

    ## Encode the Gender column
    input_df['Gender'] = gender_encoder.transform(input_df['Gender'])

    ## Encode the Geography data
    encoded_geo = geography_encoder.transform([[data['Geography']]])
    encoded_geo_df = pd.DataFrame(encoded_geo,columns=geography_encoder.get_feature_names_out(['Geography']))

    ## Drop the original Geography column and concatenate the encoded columns
    final_input_df = pd.concat([input_df.drop('Geography',axis=1),encoded_geo_df],axis=1)

    print("Final input DataFrame after encoding:")
    display(final_input_df)

    ## scale the data
    scaled_input_df = scaler.transform(final_input_df)

    return scaled_input_df

## Preprocess the input data with the function
preprocessed_data = preprocess_input(input_data)

Final input DataFrame after encoding:


/home/prashant/.conda/envs/deeplearning/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [48]:
probability = best_model.predict(preprocessed_data)
probability

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 196ms/step


array([[0.21755335]], dtype=float32)

In [49]:
## Predicting the output
def predict_churn(model, preprocessed_data):
    '''Predict churn using the preprocessed data.
    Args:
        model (tf.keras.Model): Trained Keras model for prediction.
        preprocessed_data (np.ndarray): Preprocessed input data.
    Returns:
        str: Prediction result indicating whether the customer will churn or not.
    '''
    prediction = model.predict(preprocessed_data)
    print(f"Raw model prediction output: {prediction}")

    ## Convert probability to class label
    predicted_class = (prediction > 0.5).astype(int)

    return "Customer will Churn" if predicted_class[0][0] > 0.5 else "Customer will Not Churn"


predict_churn(best_model, preprocessed_data)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
Raw model prediction output: [[0.21755335]]


'Customer will Not Churn'